## 환경 설정 및 라이브러리 임포트

In [ ]:
# !pip install augraphy
# !pip install git+https://github.com/NVlabs/ocrodeg

import os
import cv2
import numpy as np
import ocrodeg
import random
from augraphy import (
    AugraphyPipeline, LightingGradient, ShadowCast,
    Jpeg, SubtleNoise, Brightness, ColorPaper, BadPhotoCopy, OneOf,
)
import matplotlib.pyplot as plt

print("로드 완료")

## 경로 설정

In [ ]:
BASE_DIR   = "../../data/ocr/prescriptions"
INPUT_DIR  = os.path.join(BASE_DIR, "output_png")
OUTPUT_DIR = os.path.join(BASE_DIR, "output_aug")
os.makedirs(OUTPUT_DIR, exist_ok=True)

AUG_PER_IMAGE = 5

print(f"입력: {INPUT_DIR}")
print(f"출력: {OUTPUT_DIR}")

## 파이프라인 정의

In [ ]:
def build_augraphy_pipeline():
    ink_phase = [
        OneOf([
            BadPhotoCopy(
                noise_mask=None, noise_type=-1, noise_side="random",
                noise_iteration=(1, 2), noise_size=(1, 2),
                noise_value=(0, 10),
                noise_sparsity=(0.95, 0.99),
                noise_concentration=(0.05, 0.15),
                blur_noise=True,
                blur_noise_kernel=(3, 3), wave_pattern=False,
                edge_effect=False, p=0.3,
            ),
        ], p=0.2),
    ]
    paper_phase = [
        ColorPaper(hue_range=(20, 40), saturation_range=(5, 20), p=0.5),
        Brightness(brightness_range=(0.85, 1.1), min_brightness=0, min_brightness_value=(120, 150), p=0.6),
    ]
    post_phase = [
        LightingGradient(light_position=None, direction=None, max_brightness=196, min_brightness=96, mode="gaussian", transparency=0.5, p=0.8),
        ShadowCast(shadow_side="random", shadow_vertices_range=(2, 3), shadow_width_range=(0.3, 0.6), shadow_height_range=(0.3, 0.6), shadow_color=(0,0,0), shadow_opacity_range=(0.1, 0.3), shadow_iterations_range=(1, 2), shadow_blur_kernel_range=(101, 201), p=0.4),
        Jpeg(quality_range=(70, 95), p=0.7),
        SubtleNoise(subtle_range=8, p=0.5),
    ]
    return AugraphyPipeline(ink_phase=ink_phase, paper_phase=paper_phase, post_phase=post_phase, save_outputs=False)


def apply_ocrodeg_curl(img):
    curl_strength = random.uniform(1, 5)
    channels = []
    for c in range(3):
        ch = img[:, :, c].astype(np.float32) / 255.0
        noise = ocrodeg.bounded_gaussian_noise(ch.shape, curl_strength, curl_strength * 0.3)
        distorted = ocrodeg.distort_with_noise(ch, noise)
        channels.append(np.clip(distorted * 255, 0, 255).astype(np.uint8))
    return cv2.merge(channels)


def augment(img):
    pipeline = build_augraphy_pipeline()
    result = pipeline(img)
    if random.random() < 0.5:
        result = apply_ocrodeg_curl(result)
    return result

print("파이프라인 정의 완료")

## 단건 테스트 (처방전_0001.png)

In [ ]:
test_path = os.path.join(INPUT_DIR, "처방전_0001.png")
img = cv2.imdecode(np.fromfile(test_path, dtype=np.uint8), cv2.IMREAD_COLOR)

fig, axes = plt.subplots(1, 4, figsize=(24, 8))
axes[0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
axes[0].set_title("원본", fontsize=12)
axes[0].axis("off")

for i in range(3):
    result = augment(img)
    axes[i+1].imshow(cv2.cvtColor(result, cv2.COLOR_BGR2RGB))
    axes[i+1].set_title(f"증강 #{i+1}", fontsize=12)
    axes[i+1].axis("off")

save_path = os.path.join(OUTPUT_DIR, "처방전_0001_test_preview.png")
plt.savefig(save_path, dpi=150, bbox_inches="tight")
plt.tight_layout()
plt.show()
print(f"완료 → {save_path}")

## 전체 배치 증강

In [ ]:
png_files = sorted([f for f in os.listdir(INPUT_DIR) if f.endswith(".png")])
print(f"총 {len(png_files)}개 파일")

total = 0
for fname in png_files:
    base = os.path.splitext(fname)[0]
    img = cv2.imdecode(np.fromfile(os.path.join(INPUT_DIR, fname), dtype=np.uint8), cv2.IMREAD_COLOR)

    for k in range(AUG_PER_IMAGE):
        result = augment(img)
        out_path = os.path.join(OUTPUT_DIR, f"{base}_aug{k+1:02d}.png")
        cv2.imencode('.png', result)[1].tofile(out_path)
        total += 1

    print(f"완료: {fname} → {AUG_PER_IMAGE}장")

print(f"\n총 {total}장 완료 → {OUTPUT_DIR}")